# Inspecting both All_beautry.jsonl and meta_All_Beauty.jsonl

In [ ]:
import pandas as pd
import re

reviews = pd.read_json(
    "../data/raw/All_Beauty.jsonl",
    lines=True,
    nrows=5,
)

reviews

In [ ]:
metadata = pd.read_json(
    "../data/raw/meta_All_Beauty.jsonl",
    lines=True,
    nrows=5,
)

metadata

In [ ]:
reviews.columns

metadata.columns

In [ ]:
reviews.info()


metadata.info()

In [ ]:
reviews = pd.read_json(
    "../data/raw/All_Beauty.jsonl",
    lines=True, 
)

metadata = pd.read_json(
    "../data/raw/meta_All_Beauty.jsonl",
    lines=True,
)

In [ ]:
print("Reviews shape:", reviews.shape)
print("Metadata shape:", metadata.shape)


reviews.isna().sum()
metadata.isna().sum()

reviews["rating"].value_counts().sort_index()
reviews["rating"].value_counts(normalize=True).sort_index()

In [ ]:
print("Unique parent_asin in reviews:", reviews["parent_asin"].nunique())
print("Unique parent_asin in metadata:", metadata["parent_asin"].nunique())

print()

reviews["parent_asin"].isna().sum()
metadata["parent_asin"].isna().sum()

In [ ]:
print("Missing values in reviews:")
print(reviews.isna().sum())

print("\nMissing values in metadata:")
print(metadata.isna().sum())

In [ ]:
empty_text = reviews["text"].str.strip().eq("").sum()

print("Empty review texts:", empty_text)

In [ ]:
reviews["text_length"] = reviews["text"].str.len()

reviews["text_length"].describe()

In [ ]:
print("Reviews shorter than 10 characters:",
      (reviews["text_length"] < 10).sum())

print("Reviews shorter than 20 characters:",
      (reviews["text_length"] < 20).sum())

In [ ]:
reviews.loc[reviews["text_length"] < 20, ["rating", "text"]].head(20)

In [ ]:
review_products = set(reviews["parent_asin"])
metadata_products = set(metadata["parent_asin"])

matching_products = review_products & metadata_products

print("Unique products in reviews:", len(review_products))
print("Unique products in metadata:", len(metadata_products))
print("Matching products:", len(matching_products))


print("Review products with metadata:", len(matching_products) / len(review_products))

In [ ]:
print("Duplicate review texts:", reviews["text"].duplicated().sum())

duplicate_reviews = reviews.duplicated(subset=["user_id", "parent_asin", "rating", "text"]).sum()

print("Potential duplicate reviews:", duplicate_reviews)

In [ ]:
reviews.loc[
    reviews.duplicated(
        subset=["user_id", "parent_asin", "rating", "text"],
        keep=False),
        ["user_id", "parent_asin", "rating", "text"]
        ].head(20)

In [ ]:
duplicate_reviews_asin = reviews.duplicated(subset=["user_id", "asin", "rating", "text"]).sum()

print("Potential duplicate reviews using asin:", duplicate_reviews_asin)

In [ ]:
exact_review_duplicates = reviews.duplicated(
    subset=[
        "user_id",
        "asin",
        "rating",
        "title",
        "text",
        "timestamp"
    ]
).sum()

print("More exact duplicate reviews:", exact_review_duplicates)

In [ ]:
reviews.loc[
    reviews.duplicated(
        subset=[
            "user_id",
            "asin",
            "rating",
            "title",
            "text",
            "timestamp"
        ],
        keep=False
    ),
    [
        "user_id",
        "asin",
        "parent_asin",
        "rating",
        "title",
        "text",
        "timestamp"
    ]
].head(20)

In [ ]:
duplicate_groups = (
    reviews.groupby(["user_id", "asin", "rating", "title", "text", "timestamp"]).size().sort_values(ascending=False)
)

duplicate_groups.head(20)

In [ ]:
def assign_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

reviews["sentiment"] = reviews["rating"].apply(assign_sentiment)

reviews["sentiment"].value_counts()

In [ ]:
reviews["sentiment"].value_counts(normalize=True)

In [ ]:
cleaned_reviews = reviews.copy()

# Remove reviews with empty or whitespace-only text
cleaned_reviews = cleaned_reviews[
    cleaned_reviews["text"].str.strip().ne("")
].copy()

# Remove exact duplicate reviews
cleaned_reviews = cleaned_reviews.drop_duplicates(
    subset=["user_id", "asin", "rating", "title", "text", "timestamp"]
).copy()

# Columns used to identify exact duplicate reviews
duplicate_columns = [
    "user_id",
    "asin",
    "rating",
    "title",
    "text",
    "timestamp"
]

# Remove exact duplicate reviews
cleaned_reviews = cleaned_reviews.drop_duplicates(
    subset=duplicate_columns
).copy()

# Verify that exact duplicates have been removed
print("Exact duplicates after cleaning:", cleaned_reviews.duplicated(
    subset=duplicate_columns
).sum())

# Keep only columns needed later in the project
cleaned_reviews = cleaned_reviews[
    [
        "rating",
        "title",
        "text",
        "asin",
        "parent_asin",
        "sentiment",
    ]
].copy()

cleaned_reviews = cleaned_reviews.reset_index(drop=True)

# Testing new processed data in data/processed

In [ ]:
saved_cleaned = pd.read_csv(
    "../data/processed/all_beauty_cleaned.csv",
    keep_default_na=False
)

saved_sample = pd.read_csv(
    "../data/processed/all_beauty_sample_100k.csv",
    keep_default_na=False
)

print("Saved cleaned shape:", saved_cleaned.shape)
print("Saved sample shape:", saved_sample.shape)

In [ ]:
print("Missing values in cleaned data:")
print(saved_cleaned.isna().sum())

print("\nMissing values in sample data:")
print(saved_sample.isna().sum())

In [ ]:
print("Sample sentiment counts:")
print(saved_sample['sentiment'].value_counts())

print("\nSample sentiment proportions:")
print(saved_sample['sentiment'].value_counts(normalize=True))

In [ ]:
star_patterns = [
    r"\b1 star\b",
    r"\b2 stars?\b",
    r"\b3 stars?\b",
    r"\b4 stars?\b",
    r"\b5 stars?\b",
    r"\bone star\b",
    r"\btwo stars?\b",
    r"\bthree stars?\b",
    r"\bfour stars?\b",
    r"\bfive stars?\b",
]

pattern = "|".join(star_patterns)

explicit_star_mentions = reviews[
    reviews["text"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False,
    )
]

print("Reviews with explicit star mentions:", len(explicit_star_mentions))

9825 reviews contain explicit star-rating mentions in the review text.  
These are not removed during data cleaning at this stage.  
Not sure how to handle them before model training (group decision), because they may reveal the target class.  

In [ ]:
explicit_star_mentions[
    ["rating", "title", "text"]
].head(20)

In [ ]:
print(
    explicit_star_mentions["rating"]
    .value_counts()
    .sort_index()
)

In [ ]:
star_number_pattern = (
    r"\b([1-5])\s+stars?\b"
    r"|\b(one|two|three|four|five)\s+stars?\b"
)

star_mentions_check = reviews[
    reviews["text"].str.contains(
        star_number_pattern,
        case=False,
        regex=True,
        na=False,
    )
].copy()

In [ ]:
word_to_number = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
}

extracted = star_mentions_check["text"].str.extract(
    star_number_pattern,
    flags=re.IGNORECASE,
)

star_mentions_check["mentioned_rating"] = (
    extracted[0]
    .fillna(extracted[1].str.lower().map(word_to_number))
    .astype(int)
)

In [ ]:
star_mentions_check["rating_matches_text"] = (
    star_mentions_check["rating"]
    == star_mentions_check["mentioned_rating"]
)

print(
    star_mentions_check["rating_matches_text"]
    .value_counts()
)

print(
    star_mentions_check["rating_matches_text"]
    .value_counts(normalize=True)
)

In [ ]:
explicit_star_mentions_title = reviews[
    reviews["title"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False,
    )
]

print(
    "Titles with explicit star mentions:",
    len(explicit_star_mentions_title)
)

In [ ]:
star_mentions_title_check = reviews[
    reviews["title"].str.contains(
        star_number_pattern,
        case=False,
        regex=True,
        na=False,
    )
].copy()

extracted_title = star_mentions_title_check["title"].str.extract(
    star_number_pattern,
    flags=re.IGNORECASE,
)

star_mentions_title_check["mentioned_rating"] = (
    extracted_title[0]
    .fillna(extracted_title[1].str.lower().map(word_to_number))
    .astype(int)
)

star_mentions_title_check["rating_matches_title"] = (
    star_mentions_title_check["rating"]
    == star_mentions_title_check["mentioned_rating"]
)

print(
    star_mentions_title_check["rating_matches_title"]
    .value_counts()
)

print(
    star_mentions_title_check["rating_matches_title"]
    .value_counts(normalize=True)
)

### Explicit star-rating mentions

Star-rating mentions were found in both review text and review titles.

- In review `text`, the mentioned star rating matched the actual rating in about **64.7%** of matched reviews.
- In review `titles`, the mentioned star rating matched the actual rating in about **99.4%** of matched reviews.

Because titles often directly reveal the rating, `title` should not be used as a model feature in its current form.  

The review `text` column are a risk and a potential leakage risk and can be handled later if needed, but doesn't always reveal the rating.  
What this means for `text` is that of about 701 528 reviews, 9825 mention a rating in their review `text`, of those about 6368 is the actual rating. Which is only about 0.9% of all reviews in total.
